# Parametric Testing Cheat Sheet

A quick-reference companion to the extended lab. Skim the decision table, then copy-paste the snippet you need.


## 1. Which test do I use?

| Question | Test | Assumptions |
|---|---|---|
| Is one sample's mean different from a known value? | One-sample t-test | Normal, independent |
| Are two independent group means different, equal variance? | Pooled two-sample t-test | Normal, independent, equal variance |
| Are two independent group means different, unequal variance? | Welch's t-test | Normal, independent |
| Are two *paired* measurements different (pre/post)? | Paired t-test | Differences normal, independent |
| Are 3+ independent group means different (equal variance)? | One-way ANOVA (`use_var='equal'`) | Normal, independent, equal variance |
| Are 3+ independent group means different (unequal variance)? | Welch's ANOVA (`use_var='unequal'`, statsmodels default) | Normal, independent |
| Which specific pairs differ after a significant ANOVA? | Tukey's HSD | Same as ANOVA |
| Running several t-tests at once — how do I control false positives? | Bonferroni correction (or another `multipletests` method) | — |
| Is there a linear relationship between two continuous variables? | Pearson's r | Independent, paired, continuous, finite variance (no normality/equal-variance requirement) |
| How big a sample do I need / how much power do I have? | Power analysis (`TTestPower`, `TTestIndPower`) | Matches the test you're powering |

## 2. Assumption checks — one-liners

**Normality (visual):**
```python
import scipy.stats as stats
stats.probplot(data, dist="norm", plot=plt)   # QQ plot
plt.hist(data)                                 # histogram
```

**Normality (formal tests):**
```python
# Kolmogorov-Smirnov -- MUST center (mean 0) and scale (sd 1) first
x_scaled = (data - data.mean()) / data.std()
stats.kstest(x_scaled, stats.norm.cdf)

# Anderson-Darling -- weights tails more heavily; compare statistic to critical_values
result = stats.anderson(data, dist='norm')

# Shapiro-Wilk -- best for small samples (n < ~50)
stats.shapiro(data)
```
| Test | Best for | Sensitive to |
|---|---|---|
| Kolmogorov-Smirnov | Large samples | Center of distribution |
| Anderson-Darling | Large samples, heavy tails | Tails |
| Shapiro-Wilk | Small samples (n < 50) | General (most robust choice for small n) |

**Independence (serial correlation):**
```python
from statsmodels.stats.stattools import durbin_watson
durbin_watson(data)   # ~2 = no autocorrelation, ~0 = positive, ~4 = negative
```
Independence from *sampling design* (no subgroup leakage) cannot be tested from the data alone -- it's a design/methodology check.

**Equal variance:**
```python
stats.levene(group1, group2, group3)   # 2+ groups

# Fisher's F-test, 2 groups only (larger-variance group as numerator)
f_stat = np.var(bigger, ddof=1) / np.var(smaller, ddof=1)
p = 2 * min(stats.f.cdf(f_stat, df1, df2), 1 - stats.f.cdf(f_stat, df1, df2))
```


## 3. t-tests — syntax

```python
import scipy.stats as stats
from statsmodels.stats.weightstats import ttest_ind as sm_ttest_ind

# One-sample
stats.ttest_1samp(data, popmean=100, alternative='greater')  # 'two-sided' | 'less' | 'greater'

# Two-sample, pooled (equal variance)
sm_ttest_ind(group_a, group_b, alternative='two-sided', usevar='pooled')

# Two-sample, Welch's (unequal variance)
stats.ttest_ind(group_a, group_b, equal_var=False)

# Paired
stats.ttest_rel(post, pre, alternative='greater')
```

**Critical values / manual p-values:**
```python
stats.t.ppf(q=alpha, df=df)          # critical value, left-tailed
stats.t.sf(abs(t_stat), df=df)       # one-tailed p-value from a t-statistic
stats.t.sf(abs(t_stat), df=df) * 2   # two-tailed p-value
```

**Effect size (Cohen's d):**
```python
# one-sample
d = (sample_mean - hypothesized_mean) / sample_std

# two independent samples (pooled sd)
pooled_sd = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
d = (mean1 - mean2) / pooled_sd

# paired
d = mean_of_differences / sd_of_differences
```
| |d| | Interpretation |
|---|---|
| ~0.2 | small |
| ~0.5 | medium |
| ~0.8 | large |


## 4. Multiple tests & ANOVA — syntax

```python
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Bonferroni (or 'holm', 'fdr_bh', etc.)
reject, corrected_p, *_ = multipletests(pvalues, alpha=0.05, method='bonferroni')

# One-way ANOVA
anova_oneway(data['y'], data['group'], use_var='equal')     # classic Fisher
anova_oneway(data['y'], data['group'], use_var='unequal')   # Welch's (default)

# ANOVA via a fitted linear model (gives an ANOVA table)
model = ols('y ~ C(group)', data=data).fit()
anova_lm(model)

# Post-hoc pairwise comparisons that control the FWER directly
tukey = pairwise_tukeyhsd(endog=data['y'], groups=data['group'], alpha=0.05)
print(tukey)
```

**Bonferroni formula:** for `m` tests and family-wise error rate α, each individual test needs `p_i <= alpha/m` to be significant. It's conservative (increases Type II error) -- consider `method='holm'` or `method='fdr_bh'` for more power when `m` is large.


## 5. Pearson's correlation — syntax

```python
from scipy.stats import pearsonr
r, p_value = pearsonr(x, y)
r_squared = r ** 2   # proportion of variance explained ("goodness of fit")
```
| |r| | Interpretation |
|---|---|
| 0.0 - 0.1 | negligible |
| 0.1 - 0.3 | weak |
| 0.3 - 0.5 | moderate |
| 0.5 - 1.0 | strong |

Requires: independent + paired observations, continuous data, finite variance. Does **not** require normality or equal variance. Only captures **linear** association -- always look at the scatterplot.


## 6. Power analysis — syntax

```python
from statsmodels.stats.power import TTestPower, TTestIndPower

# One-sample / paired
TTestPower().solve_power(effect_size=d, nobs=n, alpha=0.05, alternative='two-sided')       # -> power
TTestPower().solve_power(effect_size=d, power=0.80, alpha=0.05, alternative='two-sided')   # -> required n

# Two independent samples
TTestIndPower().solve_power(effect_size=d, nobs1=n, alpha=0.05, ratio=1.0)                 # -> power
TTestIndPower().solve_power(effect_size=d, power=0.80, alpha=0.05, ratio=1.0)               # -> required n per group
```
`solve_power` always solves for whichever argument you omit -- pass exactly 3 of {effect_size, nobs, alpha, power} and it computes the 4th.


## 7. Common pitfalls

| Pitfall | Why it matters |
|---|---|
| Running KS test without centering/scaling first | KS test as implemented in scipy assumes the reference distribution is standard normal; un-scaled data will almost always "fail" |
| Using Shapiro-Wilk on very large samples | Nearly any large sample will show *some* p < 0.05 deviation from perfect normality (over-powered); use visual inspection + effect-size judgment too |
| Treating a large-sample significant p-value as "important" | With big n, tiny/trivial effects become "significant" — always report effect size alongside p-value |
| Running many pairwise t-tests with no correction | Inflates family-wise Type I error — use Bonferroni/Holm/Tukey |
| Assuming ANOVA tells you *which* groups differ | ANOVA only tests "not all means are equal" — follow up with a post-hoc test |
| Using Pearson's r on a non-linear relationship | r can be near 0 even with a strong non-linear (e.g. U-shaped) relationship — plot the scatter first |
| Confusing correlation direction assumptions | Pearson's r doesn't assume an input/output variable — it's symmetric between x and y |
| Applying Durbin-Watson to non-sequential data | Only meaningful when row order reflects a real sequence (time, trial number) |


## 8. Significance level guidance

| Confidence level | alpha | Typical use |
|---|---|---|
| 90% | 0.10 | Exploratory analysis, lenient screening |
| 95% | 0.05 | Standard default in most fields |
| 99% | 0.01 | High-stakes decisions, multiple-testing-heavy contexts |

If normality is only approximate (not clearly violated, not clearly satisfied), the chapter's guidance is to consider a *more lenient* confidence level (e.g., 90% instead of 99%) rather than abandoning the parametric test outright.
